In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import ParameterGrid


In [16]:
df = pd.read_csv("..\Snowflake Notebooks Package\data\data_join.csv")

print(df.shape)
df.head()



(9319, 13)


,LATITUDE,LONGITUDE,DATE,TOTALAL_KALINITY,ELECTRICAL_CONDUCTANCE,DISSOLVED_REACTIVE_PHOSPHORUS,PET,NIR,GREEN,SWIR16,SWIR22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.2,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.1,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.5,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.7,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.2,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [17]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%d-%m-%Y")
df = df.sort_values("DATE").reset_index(drop=True)
df["month"] = df["DATE"].dt.month
df["year"] = df["DATE"].dt.year
df["dayofyear"] = df["DATE"].dt.dayofyear

In [18]:
numeric_cols = [
    "Latitude", "Longitude",
    "NIR", "GREEN", "SWIR16", "SWIR22",
    "NDMI", "MNDWI", "PET",
    "Total alkalinity", "Electrical conducata", "Dissolved reactive phosphourus"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [19]:
eps = 1e-6

df["water_moisture_index"] = df["NDMI"] * df["MNDWI"]
df["evaporation_stress"] = df["PET"] / (df["NDMI"].abs() + eps)
df["SWIR_ratio"] = df["SWIR16"] / (df["SWIR22"] + eps)
df["SWIR_diff"] = df["SWIR16"] - df["SWIR22"]
df["is_water"] = (df["MNDWI"] > 0).astype(int)

df["nir_green_ratio"] = df["NIR"] / (df["GREEN"] + eps)
df["nir_swir16_ratio"] = df["NIR"] / (df["SWIR16"] + eps)
df["nir_swir22_ratio"] = df["NIR"] / (df["SWIR22"] + eps)
df["green_swir16_ratio"] = df["GREEN"] / (df["SWIR16"] + eps)
df["green_swir22_ratio"] = df["GREEN"] / (df["SWIR22"] + eps)
df["swir16_swir22_ratio"] = df["SWIR16"] / (df["SWIR22"] + eps)

df["lat_ndmi"] = df["LATITUDE"] * df["NDMI"]
df["lon_ndmi"] = df["LONGITUDE"] * df["NDMI"]
df["lat_pet"] = df["LATITUDE"] * df["PET"]
df["lon_pet"] = df["LONGITUDE"] * df["PET"]

df["nir_minus_green"] = df["NIR"] - df["GREEN"]
df["nir_minus_swir16"] = df["NIR"] - df["SWIR16"]
df["nir_minus_swir22"] = df["NIR"] - df["SWIR22"]
df["green_minus_swir16"] = df["GREEN"] - df["SWIR16"]
df["green_minus_swir22"] = df["GREEN"] - df["SWIR22"]

df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

df["pet_ndmi_ratio"] = df["PET"] / (df["NDMI"] + 1e-6)
df["pet_mndwi_ratio"] = df["PET"] / (df["MNDWI"] + 1e-6)

df["pet_ndmi_product"] = df["PET"] * df["NDMI"]
df["pet_mndwi_product"] = df["PET"] * df["MNDWI"]

df["doy_sin"] = np.sin(2*np.pi*df["dayofyear"]/365)
df["doy_cos"] = np.cos(2*np.pi*df["dayofyear"]/365)

df["nir_green"] = df["NIR"] / (df["GREEN"] + 1e-6)
df["green_nir"] = df["GREEN"] / (df["NIR"] + 1e-6)

df["nir_swir_sum"] = df["NIR"] / (df["SWIR16"] + df["SWIR22"] + 1e-6)
df["swir_ratio2"] = df["SWIR22"] / (df["SWIR16"] + 1e-6)

df["nir_green_diff"] = df["NIR"] - df["GREEN"]
df["swir_diff"] = df["SWIR22"] - df["SWIR16"]

df["lat_lon_interaction"] = df["LATITUDE"] * df["LONGITUDE"]
df["lat_plus_lon"] = df["LATITUDE"] + df["LONGITUDE"]
df["lat_minus_lon"] = df["LATITUDE"] - df["LONGITUDE"]
df["lat_squared"] = df["LATITUDE"] ** 2
df["lon_squared"] = df["LONGITUDE"] ** 2

df["lat_centered"] = df["LATITUDE"] - df["LATITUDE"].mean()
df["lon_centered"] = df["LONGITUDE"] - df["LONGITUDE"].mean()
df["distance_from_center"] = np.sqrt(df["lat_centered"]**2 + df["lon_centered"]**2)

df["drought_index_1"] = df["PET"] / (df["NDMI"] + 1.5)
df["drought_index_2"] = df["PET"] * (1 - df["NDMI"])
df["drought_index_3"] = df["PET"] / (df["water_moisture_index"].abs() + 1e-3)
df["moisture_stress_interaction"] = df["NDMI"] * df["PET"]
df["water_drought_balance"] = df["MNDWI"] - (df["PET"] / (df["PET"].max() + eps))

df["year"] = df["DATE"].dt.year
df["month"] = df["DATE"].dt.month
df["day"] = df["DATE"].dt.day
df["dayofyear"] = df["DATE"].dt.dayofyear

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)


In [20]:
features = [
    "LATITUDE", "LONGITUDE",
    "NIR", "GREEN", "SWIR16", "SWIR22",
    "NDMI", "MNDWI", "PET",

    "water_moisture_index",
    "evaporation_stress",
    "SWIR_ratio",
    "SWIR_diff",
    "is_water",

    "nir_green_ratio",
    "nir_swir16_ratio",
    "nir_swir22_ratio",
    "green_swir16_ratio",
    "green_swir22_ratio",
    "swir16_swir22_ratio",

    "nir_minus_green",
    "nir_minus_swir16",
    "nir_minus_swir22",
    "green_minus_swir16",
    "green_minus_swir22",

    "lat_lon_interaction",
    "lat_plus_lon",
    "lat_minus_lon",
    "lat_squared",
    "lon_squared",
    "lat_centered",
    "lon_centered",
    "distance_from_center",

    "drought_index_1",
    "drought_index_2",
    "drought_index_3",
    "moisture_stress_interaction",
    "water_drought_balance",

    "year",
    "month",
    "day",
    "dayofyear",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos"
]

targets = [
    "TOTALAL_KALINITY",
    "ELECTRICAL_CONDUCTANCE",
    "DISSOLVED_REACTIVE_PHOSPHORUS"
]

features = [col for col in features if col in df.columns]

needed_cols = features + targets
df_model = df.dropna(subset=needed_cols).copy()


X = df_model[features]
y = df_model[targets]

In [21]:
param_grid = {
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [300, 500],
    "l2_leaf_reg": [3, 5, 7]
}

In [22]:


tscv = TimeSeriesSplit(n_splits=5)


In [23]:
print("len(df) =", len(df))

print("len(df_model) =", len(df_model))
print("df_model index head:", df_model.index[:10].tolist())
print("df_model index tail:", df_model.index[-10:].tolist())

print("len(X) =", len(X))
print("X index head:", X.index[:10].tolist())
print("X index tail:", X.index[-10:].tolist())

for target in targets:
    y = df_model[target].reset_index(drop=True)
    print(f"{target} -> len(y) = {len(y)}")

len(df) = 9319
len(df_model) = 8234
df_model index head: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10]
df_model index tail: [9309, 9310, 9311, 9312, 9313, 9314, 9315, 9316, 9317, 9318]
len(X) = 8234
X index head: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10]
X index tail: [9309, 9310, 9311, 9312, 9313, 9314, 9315, 9316, 9317, 9318]
TOTALAL_KALINITY -> len(y) = 8234
ELECTRICAL_CONDUCTANCE -> len(y) = 8234
DISSOLVED_REACTIVE_PHOSPHORUS -> len(y) = 8234


In [24]:
needed_cols = features + targets
df_model = df.dropna(subset=needed_cols).copy().reset_index(drop=True)

X = df_model[features].copy().reset_index(drop=True)

In [ ]:
best_models = {}
best_params_per_target = {}
best_scores_per_target = {}

for target in targets:
    y = df_model[target].copy().reset_index(drop=True)

    use_log = (target == "DISSOLVED_REACTIVE_PHOSPHORUS")

    best_score = -np.inf
    best_params = None
    
    for params in ParameterGrid(param_grid):
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
            X_train = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            if use_log:
                y_train_fit = np.log1p(y_train)
            else:
                y_train_fit = y_train

            model = CatBoostRegressor(
                depth=params["depth"],
                learning_rate=params["learning_rate"],
                n_estimators=params["n_estimators"],
                l2_leaf_reg=params["l2_leaf_reg"],
                loss_function="RMSE",
                random_state=42,
                verbose=False
            )

            model.fit(X_train, y_train_fit)

            pred = model.predict(X_val)

            if use_log:
                pred = np.expm1(pred)

            r2 = r2_score(y_val, pred)
            fold_scores.append(r2)

        mean_r2 = np.mean(fold_scores)

        print(
            f"params={params} | "
            f"fold_r2={[round(s, 4) for s in fold_scores]} | "
            f"mean_r2={mean_r2:.4f}"
        )

        if mean_r2 > best_score:
            best_score = mean_r2
            best_params = params

    print(f"\nBest params for {target}: {best_params}")
    print(f"Best mean CV R2 for {target}: {best_score:.4f}")

    if use_log:
        y_fit = np.log1p(y)
    else:
        y_fit = y

    final_model = CatBoostRegressor(
        depth=best_params["depth"],
        learning_rate=best_params["learning_rate"],
        n_estimators=best_params["n_estimators"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        loss_function="RMSE",
        random_state=42,
        verbose=False
    )

    final_model.fit(X, y_fit)

    best_models[target] = final_model
    best_params_per_target[target] = best_params
    best_scores_per_target[target] = best_score


Target: TOTALAL_KALINITY
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.03, 'n_estimators': 300} | fold_r2=[0.6027, 0.6684, 0.6161, 0.6859, 0.6622] | mean_r2=0.6471
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.03, 'n_estimators': 500} | fold_r2=[0.6333, 0.7013, 0.6597, 0.7294, 0.7218] | mean_r2=0.6891
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.05, 'n_estimators': 300} | fold_r2=[0.639, 0.7076, 0.6601, 0.7408, 0.7159] | mean_r2=0.6927
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.05, 'n_estimators': 500} | fold_r2=[0.6495, 0.7235, 0.69, 0.7678, 0.7596] | mean_r2=0.7181
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.1, 'n_estimators': 300} | fold_r2=[0.63, 0.7178, 0.6991, 0.7746, 0.7712] | mean_r2=0.7185
params={'depth': 4, 'l2_leaf_reg': 3, 'learning_rate': 0.1, 'n_estimators': 500} | fold_r2=[0.627, 0.722, 0.7032, 0.783, 0.789] | mean_r2=0.7249
params={'depth': 4, 'l2_leaf_reg': 5, 'learning_rate': 0.03, 'n_estimators': 300} | f

In [ ]:
for target in targets:
    print(f"\nTarget: {target}")
    print(f"Best Params: {best_params_per_target[target]}")
    print(f"Best CV R2 : {best_scores_per_target[target]:.4f}")

overall_mean_r2 = np.mean(list(best_scores_per_target.values()))
print(f"\nOverall Average CV R2: {overall_mean_r2:.4f}")


############################################################
FINAL SUMMARY
############################################################

Target: TOTALAL_KALINITY
Best Params: {'depth': 6, 'l2_leaf_reg': 3, 'learning_rate': 0.05, 'n_estimators': 500}
Best CV R2 : 0.7290

Target: ELECTRICAL_CONDUCTANCE
Best Params: {'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.1, 'n_estimators': 500}
Best CV R2 : 0.7514

Target: DISSOLVED_REACTIVE_PHOSPHORUS
Best Params: {'depth': 6, 'l2_leaf_reg': 3, 'learning_rate': 0.03, 'n_estimators': 500}
Best CV R2 : 0.3630

Overall Average CV R2: 0.6145
